In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("../data/GUIDE_Train.csv")

print(df.shape)
df.head()

(9516837, 45)


,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04T06:05:15.000Z,7,6,InitialAccess,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,31,6,3
1,455266534868,88,326,210035,2024-06-14T03:01:25.000Z,58,43,Exfiltration,NaN,FalsePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
2,1056561957389,809,58352,712507,2024-06-13T04:52:55.000Z,423,298,InitialAccess,T1189,FalsePositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
3,1279900258736,92,32992,774301,2024-06-10T16:39:36.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
4,214748368522,148,4359,188041,2024-06-15T01:08:07.000Z,9,74,Execution,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


In [3]:
cols_to_drop = [
    'ResourceType',
    'ActionGrouped',
    'ActionGranular',
    'ThreatFamily',
    'EmailClusterId',
    'AntispamDirection',
    'Roles',
    'SuspicionLevel',
    'LastVerdict'
]

df = df.drop(columns=cols_to_drop)

In [4]:
df = df.dropna(subset=['IncidentGrade'])

In [5]:
df['MitreTechniques'] = df['MitreTechniques'].fillna("Unknown")

In [6]:
df['AlertsPerDevice'] = df.groupby('DeviceName')['DeviceName'].transform('count')
df['AlertsPerUser'] = df.groupby('AccountName')['AccountName'].transform('count')
df['AlertsPerIP'] = df.groupby('IpAddress')['IpAddress'].transform('count')

In [7]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

df['Hour'] = df['Timestamp'].dt.hour
df['Day'] = df['Timestamp'].dt.day
df['Month'] = df['Timestamp'].dt.month

df = df.drop(columns=['Timestamp'])

In [8]:
high_card_cols = [
    'AlertTitle',
    'DeviceName',
    'AccountName',
    'FileName',
    'IpAddress'
]

for col in high_card_cols:
    if col in df.columns:
        freq = df[col].value_counts()
        df[col] = df[col].map(freq)

In [9]:
y = df['IncidentGrade']
X = df.drop('IncidentGrade', axis=1)

In [10]:
X = pd.get_dummies(X, drop_first=True)

In [11]:
le = LabelEncoder()
y = le.fit_transform(y)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

In [14]:
rf_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

Accuracy: 0.8153807489318924
              precision    recall  f1-score   support

           0       0.74      0.95      0.83      8605
           1       0.86      0.70      0.77      4313
           2       0.96      0.72      0.82      6977

    accuracy                           0.82     19895
   macro avg       0.85      0.79      0.81     19895
weighted avg       0.84      0.82      0.81     19895



In [13]:
from xgboost import XGBClassifier

In [14]:
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False
)

In [15]:
xgb_model.fit(X_train, y_train)

/Users/guru/Desktop/LLM_SOC_CoPilot/venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [06:18:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [16]:
xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))

XGBoost Accuracy: 0.9073799587977391
              precision    recall  f1-score   support

           0       0.87      0.96      0.91    822164
           1       0.94      0.82      0.87    406393
           2       0.94      0.90      0.92    664543

    accuracy                           0.91   1893100
   macro avg       0.92      0.89      0.90   1893100
weighted avg       0.91      0.91      0.91   1893100



In [17]:
import joblib
import pickle  # alternative

# Save using joblib (recommended for sklearn/xgboost)
joblib.dump(xgb_model, 'soc_model.pkl')
print("✅ Model saved as 'soc_model.pkl'")

# Also save feature names (important for later!)
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, 'feature_names.pkl')
print("✅ Feature names saved")

# 2. CHECK ACCURACY ON TEST DATA (you already have this)
print("\n📊 TEST SET PERFORMANCE:")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("\nDetailed Report:")
print(classification_report(y_test, xgb_pred))

✅ Model saved as 'soc_model.pkl'
✅ Feature names saved

📊 TEST SET PERFORMANCE:
Accuracy: 0.9073799587977391

Detailed Report:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91    822164
           1       0.94      0.82      0.87    406393
           2       0.94      0.90      0.92    664543

    accuracy                           0.91   1893100
   macro avg       0.92      0.89      0.90   1893100
weighted avg       0.91      0.91      0.91   1893100



In [18]:
df = pd.read_csv("../data/GUIDE_Test.csv")

print(df.shape)
df.head()

/var/folders/jd/hl5vnd217nd2sdv824_9fllc0000gp/T/ipykernel_5738/4032570626.py:1: DtypeWarning: Columns (10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/GUIDE_Test.csv")


(4147992, 46)


,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City,Usage
0,1245540519230,657,11767,87199,2024-06-04T22:56:27.000Z,524,563,LateralMovement,T1021;T1047;T1105;T1569.002,BenignPositive,...,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630,Private
1,1400159342154,3,91158,632273,2024-06-03T12:58:26.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,0,0,NaN,Suspicious,Suspicious,242,1445,10630,Public
2,1279900255923,145,32247,131719,2024-06-08T03:20:49.000Z,2932,10807,LateralMovement,T1021;T1027.002;T1027.005;T1105,BenignPositive,...,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630,Public
3,60129547292,222,15294,917686,2024-06-12T12:07:31.000Z,0,0,InitialAccess,T1078;T1078.004,FalsePositive,...,NaN,5,66,NaN,NaN,NaN,242,1445,10630,Public
4,515396080539,363,7615,5944,2024-06-06T17:42:05.000Z,27,18,Discovery,T1087;T1087.002,BenignPositive,...,Suspicious,5,66,NaN,NaN,NaN,242,1445,10630,Public


In [19]:
cols_to_drop = [
    'ResourceType',
    'ActionGrouped',
    'ActionGranular',
    'ThreatFamily',
    'EmailClusterId',
    'AntispamDirection',
    'Roles',
    'SuspicionLevel',
    'LastVerdict'
]

df = df.drop(columns=cols_to_drop)

In [20]:
df = df.dropna(subset=['IncidentGrade'])

In [21]:
df['MitreTechniques'] = df['MitreTechniques'].fillna("Unknown")

In [22]:
df['AlertsPerDevice'] = df.groupby('DeviceName')['DeviceName'].transform('count')
df['AlertsPerUser'] = df.groupby('AccountName')['AccountName'].transform('count')
df['AlertsPerIP'] = df.groupby('IpAddress')['IpAddress'].transform('count')

In [23]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

df['Hour'] = df['Timestamp'].dt.hour
df['Day'] = df['Timestamp'].dt.day
df['Month'] = df['Timestamp'].dt.month

df = df.drop(columns=['Timestamp'])

In [24]:
high_card_cols = [
    'AlertTitle',
    'DeviceName',
    'AccountName',
    'FileName',
    'IpAddress'
]

for col in high_card_cols:
    if col in df.columns:
        freq = df[col].value_counts()
        df[col] = df[col].map(freq)

In [25]:
y = df['IncidentGrade']
X = df.drop('IncidentGrade', axis=1)

In [26]:
X = pd.get_dummies(X, drop_first=True)

In [27]:
# Align columns with training data
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0

# Keep only columns that were in training
X_test = X_test[X_train.columns]

# Encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_test_encoded = le.fit_transform(y_test)

# Make predictions
y_pred = xgb_model.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score, classification_report
print("\n📊 RESULTS ON TEST DATA:")
print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred):.4f}")
print(f"Accuracy Percentage: {accuracy_score(y_test_encoded, y_pred)*100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test_encoded, y_pred))


📊 RESULTS ON TEST DATA:
Accuracy: 0.9074
Accuracy Percentage: 90.74%

Detailed Report:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91    822164
           1       0.94      0.82      0.87    406393
           2       0.94      0.90      0.92    664543

    accuracy                           0.91   1893100
   macro avg       0.92      0.89      0.90   1893100
weighted avg       0.91      0.91      0.91   1893100



In [29]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
import numpy as np

In [30]:
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

In [31]:
kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [32]:
scores = cross_val_score(
    xgb_model,
    X,
    y,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/guru/Desktop/LLM_SOC_CoPilot/venv/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/guru/Desktop/LLM_SOC_CoPilot/venv/lib/python3.14/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
  File "/Users/guru/Desktop/LLM_SOC_CoPilot/venv/lib/python3.14/site-packages/xgboost/sklearn.py", line 1761, in fit
    raise ValueError(
    ...<2 lines>...
    )
ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got ['Beni